# Step 2: Retrieval Function

**Goal:** Build and tune the retrieval layer that sits between the user's question and the LLM.

**What we learned from Step 1:**
- Positive queries return top results at 0.28 - 0.47
- Related-but-not-exact queries (hybrid work) land at 0.38 - 0.49
- Negative queries (stock price) can pull semantically adjacent content at 0.33
- A threshold alone won't separate all cases. We need threshold + strong system prompt.

**What we're building:**
1. A retrieval function with a configurable relevance threshold
2. A way to visualise what gets through vs. what gets filtered
3. Validation against our golden dataset (retrieval only, no LLM yet)

In [2]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "gitlab_handbook"

ef = embedding_functions.DefaultEmbeddingFunction()
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"Loaded collection with {collection.count()} chunks.")

Loaded collection with 802 chunks.


## 2.1 The Retrieval Function

This is the core of the RAG pipeline. It does three things:

1. **Query ChromaDB** for the top-k most similar chunks
2. **Filter by threshold** to drop chunks that aren't similar enough
3. **Return structured output** with the context text, source files, and distances

**Why return distances?** So we can inspect them during development. In production (app.py), we won't show distances to the user, but during development they tell us whether retrieval is working.

**Threshold reasoning from Step 1 data:**
- Setting it at 0.35 (the old value) would filter out the team page results (0.47) and the hybrid work results (0.38). Too aggressive.
- Setting it at 0.80 would let everything through, including garbage. Too loose.
- **0.55 - 0.60** is the sweet spot: keeps all positive and related content, filters pure noise.

In [3]:
TOP_K = 5
RELEVANCE_THRESHOLD = 0.55  # cosine distance: lower = more similar


def retrieve(query: str, top_k: int = TOP_K, threshold: float = RELEVANCE_THRESHOLD):
    """
    Retrieve relevant chunks from ChromaDB.
    
    Returns:
        context_text: str - formatted context to feed to the LLM
        sources: list[str] - unique source filenames
        details: list[dict] - full details for inspection (distance, text preview)
    """
    results = collection.query(query_texts=[query], n_results=top_k)
    
    context_parts = []
    sources = []
    details = []
    
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]
        
        passed = distance <= threshold
        
        details.append({
            "rank": i + 1,
            "distance": distance,
            "source": source,
            "passed_threshold": passed,
            "preview": doc[:120]
        })
        
        if passed:
            context_parts.append(f"[Source: {source}]\n{doc}")
            if source not in sources:
                sources.append(source)
    
    context_text = "\n\n---\n\n".join(context_parts) if context_parts else ""
    return context_text, sources, details

## 2.2 Test Retrieval with Inspection

Let's run our test queries and see exactly what passes vs. gets filtered at the current threshold.

In [4]:
def inspect_retrieval(query: str):
    """Run retrieval and print a clear breakdown of what passed and what didn't."""
    context, sources, details = retrieve(query)
    
    passed_count = sum(1 for d in details if d["passed_threshold"])
    
    print(f"Query: '{query}'")
    print(f"Threshold: {RELEVANCE_THRESHOLD} | Passed: {passed_count}/{len(details)}")
    print(f"Sources: {sources if sources else '(none)'}")
    print("-" * 60)
    
    for d in details:
        status = "PASS" if d["passed_threshold"] else "FILTERED"
        print(f"  [{d['rank']}] {status} | dist={d['distance']:.4f} | {d['source']}")
        print(f"       {d['preview']}...")
    
    print("=" * 60 + "\n")

In [5]:
# Positive queries — all of these should PASS with relevant sources
inspect_retrieval("What does handbook first mean?")
inspect_retrieval("When should I escalate a handbook issue?")
inspect_retrieval("How do I add my photo to the team page?")
inspect_retrieval("What is GitLab's remote work policy?")

Query: 'What does handbook first mean?'
Threshold: 0.55 | Passed: 5/5
Sources: ['about_handbook-usage.md']
------------------------------------------------------------
  [1] PASS | dist=0.3118 | about_handbook-usage.md
       s to be handbook first, please refer to the [why handbook first](/handbook/about/handbook-usage/#why-handbook-first) sec...
  [2] PASS | dist=0.3576 | about_handbook-usage.md
       y add to or refactor the handbook to have a proper foundation. But, it saves time in the long run, and this communicatio...
  [3] PASS | dist=0.4210 | about_handbook-usage.md
       cations for other potentially affected processes.

Having a **"handbook first"** mentality ensures there is no duplicati...
  [4] PASS | dist=0.4781 | about_handbook-usage.md
       ecifically to mirror handbook information). This approach shows a [bias towards asynchronous communication](/handbook/va...
  [5] PASS | dist=0.4905 | about_handbook-usage.md
       You can remind other people of this by asking 

In [6]:
# The hybrid work case — should PASS with remote work content
# (the LLM will then explain that GitLab is all-remote, not hybrid)
inspect_retrieval("What is GitLab's hybrid work policy?")
inspect_retrieval("Does GitLab allow hybrid work?")

Query: 'What is GitLab's hybrid work policy?'
Threshold: 0.55 | Passed: 5/5
Sources: ['company_culture_all-remote_getting-started.md', 'company_culture_all-remote_asynchronous.md', 'business-technology_entapps-documentation_policies_gitlab-business-continuity-plan.md']
------------------------------------------------------------
  [1] PASS | dist=0.3845 | company_culture_all-remote_getting-started.md
       operate differently.

So differently, in fact, that many of GitLab's most effective processes would be discouraged or fo...
  [2] PASS | dist=0.3961 | company_culture_all-remote_asynchronous.md
       here businesses are increasingly remote. GitLab's seeks to more clearly define and operationalize asynchronous communica...
  [3] PASS | dist=0.4138 | business-technology_entapps-documentation_policies_gitlab-business-continuity-plan.md
       ---
title: "Business Continuity Plan"
controlled_document: true
tags:
  - security_policy
  - security_policy_cpir
---

...
  [4] PASS | dist=0.

In [7]:
# Negative queries — ideally these get FILTERED, but some may pass
# That's OK — the LLM system prompt handles the final call
inspect_retrieval("What is the capital of France?")
inspect_retrieval("What is GitLab's stock price?")

Query: 'What is the capital of France?'
Threshold: 0.55 | Passed: 0/5
Sources: (none)
------------------------------------------------------------
  [1] FILTERED | dist=0.8915 | company_culture_all-remote_gitlab-for-remote.md
       {{< external >}}</a>
        <button type="button" class="btn btn-lg btn-primary" data-bs-dismiss="modal"><i class="fa-s...
  [2] FILTERED | dist=0.8954 | communication_confidentiality-levels.md
       a code name for the project. For consistency and to make it easier to identify the genesis of these projects and their o...
  [3] FILTERED | dist=0.9024 | communication_chat.md
       d with a `g_`) correspond to a [DevOps Stage group](/handbook/product/categories/#hierarchy) and other [engineering depa...
  [4] FILTERED | dist=0.9128 | business-technology_engineering_infrastructure_reference-links.md
       ets mapped to the milestone for the time period that most of the work takes place. This is used as our lightweight versi...
  [5] FILTERED | dist=0.9183 

## 2.3 Threshold Tuning

If the results above don't look right, adjust `RELEVANCE_THRESHOLD` and re-run.

Use this cell to test multiple thresholds side by side on a single query.

In [8]:
def compare_thresholds(query: str, thresholds: list[float]):
    """Show how many chunks pass at different thresholds."""
    results = collection.query(query_texts=[query], n_results=TOP_K)
    distances = results["distances"][0]
    
    print(f"Query: '{query}'")
    print(f"Raw distances: {[f'{d:.4f}' for d in distances]}")
    print()
    for t in thresholds:
        passed = sum(1 for d in distances if d <= t)
        print(f"  Threshold {t:.2f} → {passed}/{len(distances)} chunks pass")
    print()


thresholds_to_test = [0.35, 0.45, 0.55, 0.65]

compare_thresholds("What does handbook first mean?", thresholds_to_test)
compare_thresholds("What is GitLab's hybrid work policy?", thresholds_to_test)
compare_thresholds("What is the capital of France?", thresholds_to_test)
compare_thresholds("What is GitLab's stock price?", thresholds_to_test)

Query: 'What does handbook first mean?'
Raw distances: ['0.3118', '0.3576', '0.4210', '0.4781', '0.4905']

  Threshold 0.35 → 1/5 chunks pass
  Threshold 0.45 → 3/5 chunks pass
  Threshold 0.55 → 5/5 chunks pass
  Threshold 0.65 → 5/5 chunks pass

Query: 'What is GitLab's hybrid work policy?'
Raw distances: ['0.3845', '0.3961', '0.4138', '0.4142', '0.4156']

  Threshold 0.35 → 0/5 chunks pass
  Threshold 0.45 → 5/5 chunks pass
  Threshold 0.55 → 5/5 chunks pass
  Threshold 0.65 → 5/5 chunks pass

Query: 'What is the capital of France?'
Raw distances: ['0.8915', '0.8954', '0.9024', '0.9128', '0.9183']

  Threshold 0.35 → 0/5 chunks pass
  Threshold 0.45 → 0/5 chunks pass
  Threshold 0.55 → 0/5 chunks pass
  Threshold 0.65 → 0/5 chunks pass

Query: 'What is GitLab's stock price?'
Raw distances: ['0.3331', '0.4275', '0.5012', '0.5037', '0.5038']

  Threshold 0.35 → 1/5 chunks pass
  Threshold 0.45 → 2/5 chunks pass
  Threshold 0.55 → 5/5 chunks pass
  Threshold 0.65 → 5/5 chunks pass



## 2.4 Validate Against Golden Dataset (Retrieval Only)

We check whether the retrieval step pulls chunks from the expected source files.

No LLM involved yet. We're just asking: does the retriever find the right documents?

In [10]:
import json

with open("golden_dataset.json", "r") as f:
    golden = json.load(f)

print(f"Loaded {len(golden)} golden dataset entries.\n")

for item in golden:
    question = item["question"]
    expected_source = item.get("source_file")
    qtype = item["type"]
    
    context, sources, details = retrieve(question)
    
    # Check if expected source was retrieved
    if qtype == "positive":
        found = any(expected_source in s for s in sources) if expected_source else False
        status = "PASS" if found else "MISS"
        print(f"[{status}] Q{item['id']}: {question}")
        print(f"       Expected: {expected_source}")
        print(f"       Got: {sources}")
    else:
        # Negative example: context should ideally be empty, but we accept it
        # as long as the LLM handles it correctly in Step 3
        has_context = bool(context)
        print(f"[NEG]  Q{item['id']}: {question}")
        print(f"       Context retrieved: {has_context} ({len(sources)} sources)")
    
    print()

Loaded 9 golden dataset entries.

[PASS] Q1: What does 'handbook first' mean at GitLab?
       Expected: about_handbook-usage.md
       Got: ['about_handbook-usage.md', 'about_direction.md', 'communication_confidentiality-levels.md']

[MISS] Q2: How should process changes be communicated at GitLab?
       Expected: about_handbook-usage.md
       Got: ['about_direction.md', 'acquisitions_acquisition-process_acquisition-process-communications.md', 'company_culture_all-remote_asynchronous.md', 'acquisitions_acquisition-process_integration.md']

[PASS] Q3: What is the GitLab Handbook and who is it for?
       Expected: about_direction.md
       Got: ['about_direction.md', 'about_handbook-usage.md']

[PASS] Q4: When should an issue be escalated via the handbook escalation process?
       Expected: about_escalation.md
       Got: ['about_maintenance.md', 'about_escalation.md']

[PASS] Q5: What is the 'Single Source of Truth' principle at GitLab?
       Expected: about_handbook-usage.md
     

## What to Check Before Moving to Step 3

1. **Positive questions:** Do the retrieved sources match `source_file` in the golden dataset? If not, either the chunking missed that content or the embedding model isn't connecting the question to the right chunks. Try rephrasing the golden dataset question or adjusting chunk size.

2. **Hybrid work case:** Are remote work chunks being retrieved? If yes, we're good. The LLM will handle the nuance in Step 3.

3. **Negative questions:** Some context being retrieved is expected ("stock price" pulls the public company page). The LLM system prompt will be responsible for not answering from irrelevant context. This is by design.

4. **Threshold value:** Based on the `compare_thresholds` output, pick the value that keeps all positive results while filtering as much noise as possible. Update `RELEVANCE_THRESHOLD` at the top if needed.

---

**Next:** Step 3 wires up the LLM. We'll build the prompt, test generation quality, and handle the edge cases (hybrid work, stock price) through prompt engineering.